<table style="width:100%">
<tr>
<td style="vertical-align:middle; text-align:left;">
<font size="2">
Supplementary code for the <a href="http://mng.bz/orYv">Build a Large Language Model From Scratch</a> book by <a href="https://sebastianraschka.com">Sebastian Raschka</a><br>
<br>Code repository: <a href="https://github.com/rasbt/LLMs-from-scratch">https://github.com/rasbt/LLMs-from-scratch</a>
</font>
</td>
<td style="vertical-align:middle; text-align:left;">
<a href="http://mng.bz/orYv"><img src="https://sebastianraschka.com/images/LLMs-from-scratch-images/cover-small.webp" width="100px"></a>
</td>
</tr>
</table>

# 처음부터 구현하는 Llama 2를 Llama 3.2로 변환하기

- 이 노트북은 [처음부터 구현한 GPT 아키텍처를 Llama 2로 변환하기](./converting-gpt-to-llama2.ipynb)의 후속 노트북으로, Meta AI의 Llama 2 아키텍처 모델을 단계별로 Llama 3, Llama 3.1, Llama 3.2로 변환합니다
- 이 노트북의 설명은 의도적으로 최소화하여 불필요하게 길어지지 않도록 하고 주요 코드에 집중합니다
- 아키텍처에 대한 더 많은 정보는 Llama 2와 Llama 3 논문을 참조하세요
 - [Llama 2: Open Foundation and Fine-Tuned Chat Models (2023)](https://arxiv.org/abs/2307.09288)
 - [The Llama 3 Herd of Models](https://arxiv.org/abs/2407.21783)

<img src="https://sebastianraschka.com/images/LLMs-from-scratch-images/bonus/gpt-to-llama/gpt2-to-llama2-llama3.webp?1">

In [ ]:
# pip install -r requirements-extra.txt

- 이 노트북에서 사용되는 패키지들:

In [ ]:
from importlib.metadata import version

pkgs = [
    "blobfile",         # 사전 훈련된 가중치 다운로드용
    "huggingface_hub",  # 사전 훈련된 가중치 다운로드용
    "tiktoken",         # 토크나이저 구현용
    "torch",            # 모델 구현용
]
for p in pkgs:
    print(f"{p} version: {version(p)}")

&nbsp;
# 1. Llama 모델 구현을 단계별로 변환하기

- LLM 아키텍처 구현이 처음이시라면 원본 GPT 아키텍처를 단계별로 구현하는 과정을 안내하는 [4장](../../ch04/01_main-chapter-code/ch04.ipynb)부터 시작하는 것을 권합니다
- [처음부터 구현한 GPT 아키텍처를 Llama 2로 변환하기](./converting-gpt-to-llama2.ipynb)에서는 RMSNorm 레이어, SiLU 및 SwiGLU 활성화 함수, RoPE(회전 위치 임베딩), SentencePiece 토크나이저와 같은 Llama 특정 구성 요소들을 구현합니다
- 이 노트북은 Llama 2 아키텍처를 가져와 다음을 통해 Llama 3 아키텍처로 변환합니다:
    1. 회전 임베딩 수정
    2. 그룹화된 쿼리 어텐션 구현
    3. GPT-4 토크나이저의 사용자 정의 버전 사용
- 나중에 Meta AI에서 공유한 원본 Llama 3 가중치를 아키텍처에 로드합니다

&nbsp;
## 1.1 Llama 2 구성 요소 재사용

- Llama 2는 실제로 위에서 언급하고 이 노트북 상단의 그림에서 보여준 바와 같이 Llama 3와 매우 유사합니다
- 이는 다음 코드를 사용하여 [Llama 2 노트북](./converting-gpt-to-llama2.ipynb)에서 여러 구성 블록을 가져올 수 있음을 의미합니다

In [ ]:
import os
import sys
import io
import nbformat
import types

def import_from_notebook():
    def import_definitions_from_notebook(fullname, names):
        current_dir = os.getcwd()
        path = os.path.join(current_dir, fullname + ".ipynb")
        path = os.path.normpath(path)

        # 노트북 로드
        if not os.path.exists(path):
            raise FileNotFoundError(f"Notebook file not found at: {path}")

        with io.open(path, "r", encoding="utf-8") as f:
            nb = nbformat.read(f, as_version=4)

        # 가져온 함수와 클래스를 저장할 모듈 생성
        mod = types.ModuleType(fullname)
        sys.modules[fullname] = mod

        # 노트북 셀을 통과하며 함수나 클래스 정의만 실행
        for cell in nb.cells:
            if cell.cell_type == "code":
                cell_code = cell.source
                for name in names:
                    # 함수나 클래스 정의 확인
                    if f"def {name}" in cell_code or f"class {name}" in cell_code:
                        exec(cell_code, mod.__dict__)
        return mod

    fullname = "converting-gpt-to-llama2"
    names = ["precompute_rope_params", "compute_rope", "SiLU", "FeedForward", "RMSNorm", "MultiHeadAttention"]

    return import_definitions_from_notebook(fullname, names)

In [ ]:
imported_module = import_from_notebook()

# precompute_rope_params를 재정의해야 합니다
# precompute_rope_params = getattr(imported_module, "precompute_rope_params", None)
compute_rope = getattr(imported_module, "compute_rope", None)
SiLU = getattr(imported_module, "SiLU", None)
FeedForward = getattr(imported_module, "FeedForward", None)
RMSNorm = getattr(imported_module, "RMSNorm", None)

# MultiHeadAttention은 비교 목적으로만 사용
MultiHeadAttention = getattr(imported_module, "MultiHeadAttention", None)

&nbsp;
## 1.2 수정된 RoPE

- Llama 3는 Llama 2와 유사한 회전 위치 임베딩(RoPE)을 사용합니다 (자세한 설명은 [RoPE 논문](https://arxiv.org/abs/2104.09864)을 참조하세요)
- RoPE 설정에는 약간의 미묘한 차이가 있습니다
 - Llama 3는 이제 최대 8,192개의 토큰을 지원하며, 이는 Llama 2(4,096개)의 두 배입니다
 - 소위 RoPE $\theta$(아래 식 참조)의 기본값이 10,000(Llama 2)에서 500,000(Llama 3)으로 다음 식에서 증가했습니다 ([RoPE 논문](https://arxiv.org/abs/2104.09864)에서 각색)

$$\Theta = \left\{\theta_i = \text{base}^{\frac{-2(i-1)}{d}}, i \in \left[1, 2, ..., d/2\right]\right\}$$

- 이러한 $\theta$ 값들은 회전 행렬에서 회전 각도를 결정하는 데 사용되는 미리 정의된 매개변수 집합이며, 여기서 $d$는 임베딩 공간의 차원입니다
- 기본값을 10,000에서 500,000으로 증가시키면 주파수(또는 회전 각도)가 차원에 걸쳐 더 천천히 감소하게 되며, 이는 더 높은 차원이 이전보다 더 큰 각도와 연관된다는 것을 의미합니다 (본질적으로, 주파수의 압축 해제입니다)
- 또한, 아래 코드에서 주파수를 조정하는 `freq_config` 섹션을 도입합니다. 그러나 Llama 3에서는 이를 필요로 하지 않으므로(Llama 3.1과 Llama 3.2에서만 필요), 나중에 이 `freq_config`를 다시 살펴보겠습니다 (기본값으로 `None`으로 설정되고 무시됩니다)

In [ ]:
import torch

def precompute_rope_params(head_dim, theta_base=10_000, context_length=4096, freq_config=None):
    assert head_dim % 2 == 0, "Embedding dimension must be even"

    # 역주파수 계산
    inv_freq = 1.0 / (theta_base ** (torch.arange(0, head_dim, 2)[: (head_dim // 2)].float() / head_dim))

    ################################ NEW ###############################################
    # 주파수 조정
    if freq_config is not None:
        low_freq_wavelen = freq_config["original_context_length"] / freq_config["low_freq_factor"]
        high_freq_wavelen = freq_config["original_context_length"] / freq_config["high_freq_factor"]

        wavelen = 2 * torch.pi / inv_freq

        inv_freq_llama = torch.where(
            wavelen > low_freq_wavelen, inv_freq / freq_config["factor"], inv_freq
        )

        smooth_factor = (freq_config["original_context_length"] / wavelen - freq_config["low_freq_factor"]) / (
            freq_config["high_freq_factor"] - freq_config["low_freq_factor"]
        )

        smoothed_inv_freq = (
            (1 - smooth_factor) * (inv_freq / freq_config["factor"]) + smooth_factor * inv_freq
        )

        is_medium_freq = (wavelen <= low_freq_wavelen) & (wavelen >= high_freq_wavelen)
        inv_freq_llama = torch.where(is_medium_freq, smoothed_inv_freq, inv_freq_llama)
        inv_freq = inv_freq_llama
    ####################################################################################


    # 위치 인덱스 생성
    positions = torch.arange(context_length)

    # 각도 계산
    angles = positions[:, None] * inv_freq[None, :]  # 모양: (context_length, head_dim // 2)

    # head_dim에 맞게 각도 확장
    angles = torch.cat([angles, angles], dim=1)  # 모양: (context_length, head_dim)

    # 사인과 코사인 미리 계산
    cos = torch.cos(angles)
    sin = torch.sin(angles)

    return cos, sin

- 요약하면, Llama 2와 비교한 Llama 3의 새로운 점은 컨텍스트 길이와 theta 기본 매개변수입니다:

In [ ]:
# RoPE 매개변수 인스턴스화

llama_2_context_len = 4096
llama_3_context_len = 8192

llama_2_theta_base = 10_000
llama_3_theta_base = 500_000

- 사용법은 Llama 2에서와 동일하게 유지됩니다:

In [ ]:
# 설정
batch_size = 2
num_heads = 4
head_dim = 16

# RoPE 매개변수 인스턴스화
cos, sin = precompute_rope_params(
    head_dim=head_dim,
    theta_base=llama_3_theta_base,
    context_length=llama_3_context_len
)

# 더미 쿼리와 키 텐서
torch.manual_seed(123)
queries = torch.randn(batch_size, num_heads, llama_3_context_len, head_dim)
keys = torch.randn(batch_size, num_heads, llama_3_context_len, head_dim)

# 회전 위치 임베딩 적용
queries_rot = compute_rope(queries, cos, sin)
keys_rot = compute_rope(keys, cos, sin)

&nbsp;
## 1.3 그룹화된 쿼리 어텐션

- 이 섹션에서는 다중 헤드 어텐션(MHA)을 그룹화된 쿼리 어텐션(GQA)이라는 대체 메커니즘으로 교체합니다
- 간단히 말해, GQA는 MHA의 보다 계산 및 매개변수 효율적인 버전으로 생각할 수 있습니다
- GQA에서는 여러 어텐션 헤드 간에 키와 값 프로젝션을 공유하여 그 수를 줄입니다
- 각 어텐션 헤드는 여전히 고유한 쿼리를 가지지만, 이러한 쿼리들은 동일한 키와 값 그룹에 어텐션합니다
- 다음은 2개의 키-값 그룹(kv-그룹)을 가진 GQA의 그림입니다:

<img src="https://sebastianraschka.com/images/LLMs-from-scratch-images/bonus/gpt-to-llama/grouped-query-attention.webp" width="500px">

- GQA의 기본 아이디어는 키-값 쌍에 어텐션하는 고유한 쿼리 그룹의 수를 줄여서, MHA에서 일부 행렬 곱셈의 크기와 매개변수 수를 모델링 성능을 크게 저하시키지 않으면서 줄이는 것입니다
- GQA 코드는 MHA와 매우 유사합니다 (아래 "NEW" 섹션을 통해 변경사항을 강조했습니다)
- 간단히 말해, GQA의 주요 변경점은 각 쿼리 그룹이 연관된 헤드 수에 맞게 반복되어야 한다는 것이며, 이는 아래에서 구현됩니다

- **또한 어텐션 클래스를 약간 재설계하여 `self.mask`로 저장하고 접근하는 대신 forward 메서드를 통해 마스크를 받도록 합니다. 이를 통해 메모리 사용량을 줄이기 위해 마스크를 즉석에서 구축할 수 있습니다. 이유를 미리 알려드리자면: Llama 3.1은 최대 128k 토큰의 시퀀스를 처리할 수 있으며, 128k × 128k 인과 마스크를 미리 계산하는 것은 극도로 메모리 집약적이므로 꼭 필요한 경우가 아니라면 이를 피합니다.**

In [ ]:
import torch.nn as nn


class GroupedQueryAttention(nn.Module):
    def __init__(
            self, d_in, d_out, num_heads,
            num_kv_groups,       # NEW
            dtype=None
        ):
        super().__init__()
        assert d_out % num_heads == 0, "d_out must be divisible by num_heads"
        assert num_heads % num_kv_groups == 0, "num_heads must be divisible by num_kv_groups"  # NEW

        self.d_out = d_out
        self.num_heads = num_heads
        self.head_dim = d_out // num_heads

        ############################# NEW  #############################
        # self.W_key = nn.Linear(d_in, d_out, bias=False, dtype=dtype)
        # self.W_value = nn.Linear(d_in, d_out, bias=False, dtype=dtype)
        self.W_key = nn.Linear(d_in, num_kv_groups * self.head_dim, bias=False, dtype=dtype)
        self.W_value = nn.Linear(d_in, num_kv_groups * self.head_dim, bias=False, dtype=dtype)
        self.num_kv_groups = num_kv_groups
        self.group_size = num_heads // num_kv_groups
        ################################################################

        self.W_query = nn.Linear(d_in, d_out, bias=False, dtype=dtype)
        self.out_proj = nn.Linear(d_out, d_out, bias=False, dtype=dtype)


    def forward(self, x, mask=None, cos=None, sin=None):
        ##################### NEW  #####################
        # forward 메서드는 이제 self.mask를 통해 접근하는 대신 `mask`를 받습니다.
        # 또한, RoPE를 위한 cos와 sin을 입력으로 받습니다
        ################################################    
        b, num_tokens, d_in = x.shape

        queries = self.W_query(x)  # 모양: (b, num_tokens, d_out)
        keys = self.W_key(x)  # 모양: (b, num_tokens, num_kv_groups * head_dim)
        values = self.W_value(x)  # 모양: (b, num_tokens, num_kv_groups * head_dim)

        # 쿼리, 키, 값 재구성
        queries = queries.view(b, num_tokens, self.num_heads, self.head_dim)

        ##################### NEW  #####################
        # keys = keys.view(b, num_tokens, self.num_heads, self.head_dim)
        # values = values.view(b, num_tokens, self.num_heads, self.head_dim)
        keys = keys.view(b, num_tokens, self.num_kv_groups, self.head_dim)
        values = values.view(b, num_tokens, self.num_kv_groups, self.head_dim)
        ################################################

        # 키, 값, 쿼리 전치
        keys = keys.transpose(1, 2)  # 모양: (b, num_kv_groups, num_tokens, head_dim)
        values = values.transpose(1, 2)  # 모양: (b, num_kv_groups, num_tokens, head_dim)
        queries = queries.transpose(1, 2)  # 모양: (b, num_heads, num_tokens, head_dim)

        ##################### NEW #####################
        # RoPE 적용
        if cos is not None:
            keys = compute_rope(keys, cos, sin)
            queries = compute_rope(queries, cos, sin)
        ################################################

        ##################### NEW  #####################
        # 헤드 수에 맞게 키와 값 확장
        # 모양: (b, num_heads, num_tokens, head_dim)

        keys = keys.repeat_interleave(self.group_size, dim=1)  # 모양: (b, num_heads, num_tokens, head_dim)
        values = values.repeat_interleave(self.group_size, dim=1)  # 모양: (b, num_heads, num_tokens, head_dim)
        # 예를 들어, dim=1(쿼리 그룹)을 따라 repeat_interleave 전:
        #   [K1, K2]
        # repeat_interleave 후 (각 쿼리 그룹이 group_size만큼 반복됨):
        #   [K1, K1, K2, K2]
        # 대신 일반적인 repeat을 사용했다면 다음과 같이 됩니다:
        #   [K1, K2, K1, K2]
        ################################################

        # 인과 마스크를 사용한 스케일드 닷-프로덕트 어텐션(즉, 셀프 어텐션) 계산
        # 모양: (b, num_heads, num_tokens, num_tokens)
        attn_scores = queries @ keys.transpose(2, 3)  # 각 헤드에 대한 내적

        ##################### NEW #####################
        # 즉석에서 마스크 생성
        if mask is None:
            mask = torch.triu(torch.ones(num_tokens, num_tokens, device=x.device, dtype=torch.bool), diagonal=1)
        ################################################
    
        # 마스크를 사용하여 어텐션 스코어 채우기
        attn_scores.masked_fill_(mask, -torch.inf)

        attn_weights = torch.softmax(attn_scores / keys.shape[-1]**0.5, dim=-1)
        assert keys.shape[-1] == self.head_dim

        # 모양: (b, num_tokens, num_heads, head_dim)
        context_vec = (attn_weights @ values).transpose(1, 2)

        # 헤드 결합, 여기서 self.d_out = self.num_heads * self.head_dim
        context_vec = context_vec.reshape(b, num_tokens, self.d_out)
        context_vec = self.out_proj(context_vec)  # 선택적 프로젝션

        return context_vec

- MHA에 대한 GQA의 매개변수 절약을 설명하기 위해, GPT 및 Llama 2 코드의 다음 다중 헤드 어텐션 예시를 고려해보세요:

In [ ]:
# 설정
batch_size = 1
context_len = 3000
max_context_len = 8192
embed_dim = 4096
num_heads = 32


example_batch = torch.randn((batch_size, context_len, embed_dim))

mha = MultiHeadAttention(
    d_in=embed_dim,
    d_out=embed_dim,
    context_length=max_context_len,
    num_heads=num_heads
)

mha(example_batch)

print("W_key:", mha.W_key.weight.shape)
print("W_value:", mha.W_value.weight.shape)
print("W_query:", mha.W_query.weight.shape)

- 이제 8개의 kv-그룹(Llama 3 8B가 사용하는 수)으로 그룹화된 쿼리 어텐션을 사용하면, 키와 값 행렬의 행 수가 4배 줄어든 것을 볼 수 있습니다 (32개 어텐션 헤드를 8개 kv-그룹으로 나누면 4가 되므로)

In [ ]:
gqa = GroupedQueryAttention(
    d_in=embed_dim,
    d_out=embed_dim,
    num_heads=num_heads,
    num_kv_groups=8,
)

gqa(example_batch)

print("W_key:", gqa.W_key.weight.shape)
print("W_value:", gqa.W_value.weight.shape)
print("W_query:", gqa.W_query.weight.shape)

- 참고로, GroupedQueryAttention을 표준 다중 헤드 어텐션과 동일하게 만들려면 쿼리 그룹 수(`num_kv_groups`)를 헤드 수(`num_heads`)와 같게 설정할 수 있습니다
- 마지막으로, 아래에서 매개변수 수를 비교해보겠습니다:

In [ ]:
print("총 매개변수 수:")

mha_total_params = sum(p.numel() for p in mha.parameters())
print(f"MHA: {mha_total_params:,}")

gqa_total_params = sum(p.numel() for p in gqa.parameters())
print(f"GQA: {gqa_total_params:,}")

In [ ]:
# 메모리 해제:
del mha
del gqa

&nbsp;
## 1.4 TransformerBlock 모듈 업데이트

- 다음으로, `TransformerBlock`을 업데이트합니다
- 여기서는 단순히 `MultiHeadAttention`을 `GroupedQueryAttention`으로 교체하고 새로운 RoPE 설정을 추가합니다
- 또한 `mask`, `cos`, `sin`을 받도록 `forward` 메서드를 수정합니다. 이러한 값들은 각 트랜스포머 블록에 대해 동일하므로, 한 번만 계산한 후 재사용할 수 있습니다

In [ ]:
class TransformerBlock(nn.Module):
    def __init__(self, cfg):
        super().__init__()
        self.att =  GroupedQueryAttention(  # MultiHeadAttention(
            d_in=cfg["emb_dim"],
            d_out=cfg["emb_dim"],
            num_heads=cfg["n_heads"],
            num_kv_groups=cfg["n_kv_groups"],  # NEW
            dtype=cfg["dtype"]
        )
        self.ff = FeedForward(cfg)
        self.norm1 = RMSNorm(cfg["emb_dim"], eps=1e-5)
        self.norm2 = RMSNorm(cfg["emb_dim"], eps=1e-5)

    def forward(self, x, mask=None, cos=None, sin=None):
        ##################### NEW  #####################
        # forward 메서드는 이제 self.mask를 통해 접근하는 대신 `mask`를 받습니다.
        # 또한, RoPE를 위한 cos와 sin을 입력으로 받습니다
        ################################################
        # 어텐션 블록에 대한 바로가기 연결
        shortcut = x
        x = self.norm1(x)
        x = self.att(x.to(torch.bfloat16), mask, cos, sin)   # 모양 [batch_size, num_tokens, emb_size]
        x = x + shortcut  # 원본 입력 다시 추가

        # 피드포워드 블록에 대한 바로가기 연결
        shortcut = x
        x = self.norm2(x)
        x = self.ff(x.to(torch.bfloat16))
        x = x + shortcut  # 원본 입력 다시 추가

        return x

&nbsp;
## 1.5 모델 클래스 정의

- 모델 클래스를 설정할 때 기술적으로는 많은 작업이 필요하지 않습니다. 이름만 `Llama3Model`로 업데이트하면 됩니다
- 하지만 이제 `mask`, `cos`, `sin`을 트랜스포머 블록에 전달하므로, 여기에도 추가해야 합니다

In [ ]:
# class Llama2Model(nn.Module):
class Llama3Model(nn.Module):
    def __init__(self, cfg):
        super().__init__()
        self.tok_emb = nn.Embedding(cfg["vocab_size"], cfg["emb_dim"], dtype=cfg["dtype"])

        self.trf_blocks = nn.Sequential(
            *[TransformerBlock(cfg) for _ in range(cfg["n_layers"])])

        self.final_norm = RMSNorm(cfg["emb_dim"], eps=1e-5)
        self.out_head = nn.Linear(cfg["emb_dim"], cfg["vocab_size"], bias=False, dtype=cfg["dtype"])

        #################### NEW #####################
        cos, sin = precompute_rope_params(
            head_dim=cfg["emb_dim"] // cfg["n_heads"],
            theta_base=cfg["rope_base"],
            context_length=cfg["context_length"],
            freq_config=cfg["rope_freq"]
        )
        
        self.register_buffer("cos", cos, persistent=False)
        self.register_buffer("sin", sin, persistent=False)
        ##############################################

        self.cfg = cfg

    def forward(self, in_idx):
        tok_embeds = self.tok_emb(in_idx)
        x = tok_embeds

        #################### NEW #####################
        num_tokens = x.shape[1]
        mask = torch.triu(torch.ones(num_tokens, num_tokens, device=x.device, dtype=torch.bool), diagonal=1)
        ##############################################
        
        for block in self.trf_blocks:
            x = block(x, mask, self.cos, self.sin)
        x = self.final_norm(x)
        logits = self.out_head(x.to(self.cfg["dtype"]))
        return logits

&nbsp;
## 2. 모델 초기화

- 이제 Llama 3 구성 파일을 정의할 수 있습니다 (비교를 위해 Llama 2 구성 파일도 표시됩니다)

In [ ]:
LLAMA2_CONFIG_7B = {
    "vocab_size": 32_000,    # 어휘 크기(vocabulary size)
    "context_length": 4096,  # 컨텍스트 길이(context length)
    "emb_dim": 4096,         # 임베딩 차원(embedding dimension)
    "n_heads": 32,           # 어텐션 헤드 수(number of attention heads)
    "n_layers": 32,          # 레이어 수(number of layers)
    "hidden_dim": 11_008,    # FeedForward의 중간 차원 크기
    "dtype": torch.bfloat16  # 메모리 사용량을 줄이기 위한 낮은 정밀도 dtype
}

In [ ]:
LLAMA3_CONFIG_8B = {
    "vocab_size": 128_256,   # NEW: 더 큰 어휘 크기
    "context_length": 8192,  # NEW: 더 큰 컨텍스트 길이
    "emb_dim": 4096,         # 임베딩 차원(embedding dimension)
    "n_heads": 32,           # 어텐션 헤드 수(number of attention heads)
    "n_layers": 32,          # 레이어 수(number of layers)
    "hidden_dim": 14_336,    # NEW: FeedForward의 더 큰 중간 차원 크기
    "n_kv_groups": 8,        # NEW: 그룹화된 쿼리 어텐션을 위한 키-값 그룹
    "rope_base": 500_000.0,  # NEW: RoPE의 "theta" 기본값이 500,000으로 증가
    "rope_freq": None,       # NEW: RoPE 주파수 조정을 위한 추가 구성
    "dtype": torch.bfloat16  # 메모리 사용량을 줄이기 위한 낮은 정밀도 dtype
}

- 이러한 설정을 사용하여 이제 Llama 3 8B 모델을 초기화할 수 있습니다
- 이는 약 34GB의 메모리가 필요함에 주의하세요 (비교해보면, Llama 2 7B는 약 26GB의 메모리가 필요했습니다)

In [ ]:
model = Llama3Model(LLAMA3_CONFIG_8B)

- 이제 훈련 가능한 매개변수의 수를 계산해보겠습니다:

In [ ]:
total_params = sum(p.numel() for p in model.parameters())
print(f"Total number of parameters: {total_params:,}")

- 위에서 보듯이 모델은 80억 개의 매개변수를 포함합니다
- 또한 아래 코드를 사용하여 이 모델의 메모리 요구사항을 계산할 수 있습니다:

In [ ]:
def model_memory_size(model, input_dtype=torch.float32):
    total_params = 0
    total_grads = 0
    for param in model.parameters():
        # 매개변수당 총 요소 수 계산
        param_size = param.numel()
        total_params += param_size
        # 이 매개변수에 대해 기울기가 저장되는지 확인
        if param.requires_grad:
            total_grads += param_size

    # 버퍼 크기 계산 (메모리가 필요한 비매개변수)
    total_buffers = sum(buf.numel() for buf in model.buffers())

    # 바이트 크기 = (요소 수) * (각 요소의 바이트 크기)
    # 매개변수와 기울기가 입력 dtype과 같은 유형으로 저장된다고 가정
    element_size = torch.tensor(0, dtype=input_dtype).element_size()
    total_memory_bytes = (total_params + total_grads + total_buffers) * element_size

    # 바이트를 기가바이트로 변환
    total_memory_gb = total_memory_bytes / (1024**3)

    return total_memory_gb

print(f"float32 (PyTorch default): {model_memory_size(model, input_dtype=torch.float32):.2f} GB")
print(f"bfloat16: {model_memory_size(model, input_dtype=torch.bfloat16):.2f} GB")

- 마지막으로 해당되는 경우 NVIDIA 또는 Apple Silicon GPU로 모델을 전송할 수도 있습니다:

In [ ]:
if torch.cuda.is_available():
    device = torch.device("cuda")
elif torch.backends.mps.is_available():
    device = torch.device("mps")
else:
    device = torch.device("cpu")

model.to(device);

&nbsp;
## 3. 토크나이저 로드

- 이 섹션에서는 모델용 토크나이저를 로드할 것입니다
- Llama 2는 OpenAI의 [Tiktoken](https://github.com/openai/tiktoken) 라이브러리 기반 BPE 토크나이저 대신 Google의 [SentencePiece](https://github.com/google/sentencepiece) 토크나이저를 사용했습니다
- 하지만 Llama 3는 Tiktoken의 BPE 토크나이저를 다시 사용합니다. 구체적으로는 확장된 어휘와 함께 GPT-4 토크나이저를 사용합니다
- Meta AI의 원본 Tiktoken 적응을 공식 Llama 3 저장소 [여기](https://github.com/meta-llama/llama3/blob/main/llama/tokenizer.py)에서 찾을 수 있습니다
- 아래에서는 이 노트북에서 더 읽기 쉽고 최소화되도록 토크나이저 코드를 다시 작성했습니다 (하지만 동작은 유사해야 합니다)

In [ ]:
from pathlib import Path

import tiktoken
from tiktoken.load import load_tiktoken_bpe


class Tokenizer:
    """Llama-3 특수 ID를 추적하는 tiktoken 주변의 얇은 래퍼."""
    def __init__(self, model_path):
        if not os.path.isfile(model_path):
            raise FileNotFoundError(model_path)

        mergeable = load_tiktoken_bpe(model_path)

        # Meta의 tokenizer.json에서 하드코딩
        self.special = {
            "<|begin_of_text|>": 128000,
            "<|end_of_text|>": 128001,
            "<|start_header_id|>": 128006,
            "<|end_header_id|>": 128007,
            "<|eot_id|>": 128009,
        }
        self.special.update({f"<|reserved_{i}|>": 128002 + i
                             for i in range(256)
                             if 128002 + i not in self.special.values()})

        self.model = tiktoken.Encoding(
            name=Path(model_path).name,
            pat_str=r"(?i:'s|'t|'re|'ve|'m|'ll|'d)"
                    r"|[^\r\n\p{L}\p{N}]?\p{L}+"
                    r"|\p{N}{1,3}"
                    r"| ?[^\s\p{L}\p{N}]+[\r\n]*"
                    r"|\s*[\r\n]+"
                    r"|\s+(?!\S)"
                    r"|\s+",
            mergeable_ranks=mergeable,
            special_tokens=self.special,
        )

    def encode(self, text, bos=False, eos=False):
        ids = ([self.special["<|begin_of_text|>"]] if bos else []) \
              + self.model.encode(text)
        if eos:
            ids.append(self.special["<|end_of_text|>"])
        return ids

    def decode(self, ids):
        return self.model.decode(ids)

- Meta AI는 원본 Llama 3 모델 가중치와 토크나이저 어휘를 Hugging Face Hub에서 공유했습니다
- 먼저 Hub에서 토크나이저 어휘를 다운로드하여 위 코드에 로드할 것입니다

- Meta AI는 파일을 다운로드하기 전에 Llama 3 라이선스 조건에 동의해야 합니다. 이를 위해서는 Hugging Face Hub 계정을 만들고 [meta-llama/Meta-Llama-3-8B](https://huggingface.co/meta-llama/Meta-Llama-3-8B) 저장소를 방문하여 조건에 동의해야 합니다
- 다음으로 액세스 토큰을 만들어야 합니다. 읽기 권한이 있는 액세스 토큰을 생성하려면 우상단의 프로필 사진을 클릭하고 "Settings"를 클릭하세요


<img src="https://sebastianraschka.com/images/LLMs-from-scratch-images/bonus/gpt-to-llama/settings.webp?1" width="300px">

- 그런 다음 액세스 토큰을 만들고 복사하여 다음 코드 셀에 복사하여 붙여넣을 수 있습니다

<img src="https://sebastianraschka.com/images/LLMs-from-scratch-images/bonus/gpt-to-llama/access-token.webp?1" width="600px">

In [ ]:
from huggingface_hub import login
import json

with open("config.json", "r") as config_file:
    config = json.load(config_file)
    access_token = config["HF_ACCESS_TOKEN"]

login(token=access_token)

- Llama 3 라이선스 조건에 동의했음을 확인하는 데 필요한 액세스 토큰을 통한 로그인 후, 이제 토크나이저 어휘를 다운로드할 수 있습니다:

In [ ]:
from huggingface_hub import hf_hub_download

tokenizer_file_path = hf_hub_download(
    repo_id="meta-llama/Meta-Llama-3-8B",
    filename="original/tokenizer.model",
    local_dir="Llama-3-8B"
)

- Llama 3 파일을 사용할 때는 `blobfile` 패키지가 필요할 수 있습니다. 이 패키지는 Google Cloud Storage(GCS), Azure Blob Storage, Amazon S3와 같은 클라우드 스토리지 솔루션에 저장된 데이터셋이나 모델을 처리할 때 사용됩니다
- 아래의 `pip` 명령어 주석을 해제하고 실행하여 이 의존성을 설치할 수 있습니다

In [ ]:
# pip install blobfile

In [ ]:
tokenizer = Tokenizer(tokenizer_file_path)

- 이제 `generate` 함수를 사용하여 Llama 3 모델이 새로운 텍스트를 생성하도록 할 수 있습니다:

In [ ]:
from previous_chapters import generate, text_to_token_ids, token_ids_to_text
# `previous_chapters.py` 파일이 로컬에 없는 경우,
# `llms-from-scratch` PyPI 패키지에서 가져올 수 있습니다.
# 자세한 내용은: https://github.com/rasbt/LLMs-from-scratch/tree/main/pkg
# 예를 들어,
# from llms_from_scratch.ch05 import generate, text_to_token_ids, token_ids_to_text


torch.manual_seed(123)

token_ids = generate(
    model=model,
    idx=text_to_token_ids("Every effort", tokenizer).to(device),
    max_new_tokens=30,
    context_size=LLAMA3_CONFIG_8B["context_length"],
    top_k=1,
    temperature=0.
)

print("Output text:\n", token_ids_to_text(token_ids, tokenizer))

- 물론 위에서 볼 수 있듯이 아직 Llama 3 모델을 훈련하지 않았기 때문에 텍스트가 의미가 없습니다
- 다음 섹션에서는 수만에서 수십만 달러의 비용이 드는 직접 훈련 대신 Meta AI의 사전 훈련된 가중치를 로드합니다

&nbsp;
## 4. 사전 훈련된 가중치 로드

- 아래에서는 미세조정 이전의 간단한 텍스트 완성 모델인 ["meta-llama/Meta-Llama-3-8B"](https://huggingface.co/meta-llama/Meta-Llama-3-8B) 기본 모델을 로드합니다
- 대안으로, 다음 코드 셀의 문자열을 적절히 수정하여 지시 미세조정되고 정렬된 ["meta-llama/Meta-Llama-3-8B-Instruct"](https://huggingface.co/meta-llama/Meta-Llama-3-8B-Instruct) 모델을 로드할 수 있습니다
- 가중치 파일들을 합쳐서 약 16GB 정도 됩니다

In [ ]:
from safetensors.torch import load_file

combined_weights = {}

for i in range(1, 5):
    weights_file = hf_hub_download(
        repo_id="meta-llama/Meta-Llama-3-8B",
        filename=f"model-0000{i}-of-00004.safetensors",
        local_dir="Llama-3-8B"
    )
    current_weights = load_file(weights_file)
    combined_weights.update(current_weights)

- `weights`에는 다음 텐서들이 포함되어 있습니다 (단순성을 위해 처음 15개만 표시됨):

In [ ]:
list(combined_weights.keys())[:15]

- 다음 함수는 [5장](../01_main-chapter-code/ch05.ipynb)의 `load_weights_into_gpt` 함수를 모델로 하여 사전 훈련된 가중치를 우리의 Llama 3 모델에 로드합니다:

In [ ]:
def assign(left, right, tensor_name="unknown"):
    if left.shape != right.shape:
        raise ValueError(f"Shape mismatch in tensor '{tensor_name}'. Left: {left.shape}, Right: {right.shape}")

    if isinstance(right, torch.Tensor):
        return torch.nn.Parameter(right.clone().detach())
    else:
        return torch.nn.Parameter(torch.tensor(right))


def load_weights_into_llama(model, param_config, params):
    model.tok_emb.weight = assign(model.tok_emb.weight, params["model.embed_tokens.weight"], "model.embed_tokens.weight")

    for l in range(param_config["n_layers"]):

        # 어텐션 가중치 로드
        model.trf_blocks[l].att.W_query.weight = assign(
            model.trf_blocks[l].att.W_query.weight,
            params[f"model.layers.{l}.self_attn.q_proj.weight"],
            f"model.layers.{l}.self_attn.q_proj.weight"
        )
        model.trf_blocks[l].att.W_key.weight = assign(
            model.trf_blocks[l].att.W_key.weight,
            params[f"model.layers.{l}.self_attn.k_proj.weight"],
            f"model.layers.{l}.self_attn.k_proj.weight"
        )
        model.trf_blocks[l].att.W_value.weight = assign(
            model.trf_blocks[l].att.W_value.weight,
            params[f"model.layers.{l}.self_attn.v_proj.weight"],
            f"model.layers.{l}.self_attn.v_proj.weight"
        )
        model.trf_blocks[l].att.out_proj.weight = assign(
            model.trf_blocks[l].att.out_proj.weight,
            params[f"model.layers.{l}.self_attn.o_proj.weight"],
            f"model.layers.{l}.self_attn.o_proj.weight"
        )
        model.trf_blocks[l].norm1.weight = assign(
            model.trf_blocks[l].norm1.weight,
            params[f"model.layers.{l}.input_layernorm.weight"],
            f"model.layers.{l}.input_layernorm.weight"
        )

        # FeedForward 가중치 로드
        model.trf_blocks[l].ff.fc1.weight = assign(
            model.trf_blocks[l].ff.fc1.weight,
            params[f"model.layers.{l}.mlp.gate_proj.weight"],
            f"model.layers.{l}.mlp.gate_proj.weight"
        )
        model.trf_blocks[l].ff.fc2.weight = assign(
            model.trf_blocks[l].ff.fc2.weight,
            params[f"model.layers.{l}.mlp.up_proj.weight"],
            f"model.layers.{l}.mlp.up_proj.weight"
        )
        model.trf_blocks[l].ff.fc3.weight = assign(
            model.trf_blocks[l].ff.fc3.weight,
            params[f"model.layers.{l}.mlp.down_proj.weight"],
            f"model.layers.{l}.mlp.down_proj.weight"
        )
        model.trf_blocks[l].norm2.weight = assign(
            model.trf_blocks[l].norm2.weight,
            params[f"model.layers.{l}.post_attention_layernorm.weight"],
            f"model.layers.{l}.post_attention_layernorm.weight"
        )

    # 출력 레이어 가중치 로드
    model.final_norm.weight = assign(model.final_norm.weight, params["model.norm.weight"], "model.norm.weight")

    if "lm_head.weight" in params.keys():
        model.out_head.weight = assign(model.out_head.weight, params["lm_head.weight"], "lm_head.weight")
    else:
        model.out_head.weight = assign(model.out_head.weight, params["model.embed_tokens.weight"], "model.embed_tokens.weight")
        print("Model uses weight tying.")


load_weights_into_llama(model, LLAMA3_CONFIG_8B, combined_weights)
model.to(device);
del combined_weights  # 메모리 해제

- 다음으로 텍스트 생성을 위해 모델을 사용할 준비가 되었습니다

In [ ]:
torch.manual_seed(123)

token_ids = generate(
    model=model,
    idx=text_to_token_ids("Every effort", tokenizer).to(device),
    max_new_tokens=25,
    context_size=LLAMA3_CONFIG_8B["context_length"],
    top_k=1,
    temperature=0.
)

print("Output text:\n", token_ids_to_text(token_ids, tokenizer))

&nbsp;
## 5. 지시 미세조정 모델 사용

- 위에서는 사전 훈련된 기본 모델을 사용했습니다. 지시를 따를 수 있는 모델을 사용하려면 아래와 같이 `"meta-llama/Llama-3-8B-Instruct"` 모델을 대신 사용하세요

In [ ]:
# 메모리 해제를 위해

import gc

del model

gc.collect()  # Python 가비지 수집기 실행

if torch.cuda.is_available():
    torch.cuda.empty_cache()

- Llama 3 모델은 미세조정 중 사용된 올바른 프롬프트 템플릿과 함께 사용되어야 합니다 (7장에서 논의된 대로)
- 아래는 프롬프트 템플릿을 구성하는 Meta AI의 Llama 3 전용 [ChatFormat 코드](https://github.com/meta-llama/llama3/blob/11817d47e1ba7a4959b025eb1ca308572e0e3963/llama/tokenizer.py#L202)를 기반으로 한 토크나이저 래퍼 클래스입니다

In [ ]:
class ChatFormat:

    def __init__(self, tokenizer: Tokenizer, *,
                 default_system="You are a helpful assistant."):
        self.tok = tokenizer
        self.default_system = default_system

    def _header(self, role):
        """<|start_header_id|>role<|end_header_id|>\n\n을 인코딩"""
        return (
            [self.tok.special["<|start_header_id|>"]]
            + self.tok.encode(role)
            + [self.tok.special["<|end_header_id|>"]]
            + self.tok.encode("\n\n")
        )

    def encode(self, user_message, system_message=None):
        sys_msg = system_message if system_message is not None else self.default_system

        ids = [self.tok.special["<|begin_of_text|>"]]

        # 시스템
        ids += self._header("system")
        ids += self.tok.encode(sys_msg)
        ids += [self.tok.special["<|eot_id|>"]]

        # 사용자
        ids += self._header("user")
        ids += self.tok.encode(user_message)
        ids += [self.tok.special["<|eot_id|>"]]

        # 어시스턴트 헤더 (아직 콘텐츠 없음)
        ids += self._header("assistant")

        return ids

- 사용법은 다음과 같습니다:

In [ ]:
tokenizer = Tokenizer(tokenizer_file_path)
chat_tokenizer = ChatFormat(tokenizer)

token_ids = chat_tokenizer.encode("Hello World!")
print(token_ids)

In [ ]:
tokenizer.decode(token_ids)

- 이제 Llama 3 지시 모델을 실제로 확인해보겠습니다:

In [ ]:
combined_weights = {}

for i in range(1, 5):
    weights_file = hf_hub_download(
        repo_id="meta-llama/Meta-Llama-3-8B-Instruct",
        filename=f"model-0000{i}-of-00004.safetensors",
        local_dir="Llama-3-8B-Instruct"
    )
    current_weights = load_file(weights_file)
    combined_weights.update(current_weights)


model = Llama3Model(LLAMA3_CONFIG_8B)
load_weights_into_llama(model, LLAMA3_CONFIG_8B, combined_weights)
model.to(device)
del combined_weights  # 메모리 해제

In [ ]:
torch.manual_seed(123)

token_ids = generate(
    model=model,
    idx=text_to_token_ids("What do llamas eat?", chat_tokenizer).to(device),
    max_new_tokens=150,
    context_size=LLAMA3_CONFIG_8B["context_length"],
    top_k=1,
    temperature=0.
)

output_text = token_ids_to_text(token_ids, tokenizer)


def clean_text(text, header_end="assistant<|end_header_id|>\n\n"):
    # "<|end_header_id|>"의 첫 번째 발생 인덱스 찾기
    index = text.find(header_end)

    if index != -1:
        # "<|end_header_id|>" 이후부터 시작하는 부분 문자열 반환
        return text[index + len(header_end):].strip()  # Strip은 앞뒤 공백 제거
    else:
        # 토큰을 찾을 수 없으면 원본 텍스트 반환
        return text

print("Output text:\n", clean_text(output_text))

&nbsp;
# Llama 3.1 8B

- 초기 Llama 3 출시 몇 달 후, Meta AI는 Llama 3.1 모델 제품군으로 후속작을 발표했습니다 (자세한 내용은 공식 [Introducing Llama 3.1: Our most capable models to date](https://ai.meta.com/blog/meta-llama-3-1/) 발표 블로그 게시물을 참조하세요)
- 편리하게도, 위의 이전 Llama 3 코드를 재사용하여 Llama 3.1 8B를 구현할 수 있습니다

<img src="https://sebastianraschka.com/images/LLMs-from-scratch-images/bonus/gpt-to-llama/llama3-to-llama31.webp" width="700px">

- 아키텍처는 동일하며, 아래 구성 파일에 표시된 바와 같이 RoPE 주파수의 재조정만이 유일한 변경사항입니다

In [ ]:
LLAMA3_CONFIG_8B = {
    "vocab_size": 128_256,   # 어휘 크기(vocabulary size)
    "context_length": 8192,  # 컨텍스트 길이(context length)
    "emb_dim": 4096,         # 임베딩 차원(embedding dimension)
    "n_heads": 32,           # 어텐션 헤드 수(number of attention heads)
    "n_layers": 32,          # 레이어 수(number of layers)
    "hidden_dim": 14_336,    # FeedForward의 중간 차원 크기
    "n_kv_groups": 8,        # 그룹화된 쿼리 어텐션을 위한 키-값 그룹
    "rope_base": 500_000.0,  # RoPE의 "theta" 기본값
    "rope_freq": None,       # RoPE 주파수 조정을 위한 추가 구성
    "dtype": torch.bfloat16  # 메모리 사용량을 줄이기 위한 낮은 정밀도 dtype
}

LLAMA31_CONFIG_8B = {
    "vocab_size": 128_256,      # 어휘 크기(vocabulary size)
    "context_length": 131_072,  # NEW: 더 큰 지원 컨텍스트 길이
    "emb_dim": 4096,            # 임베딩 차원(embedding dimension)
    "n_heads": 32,              # 어텐션 헤드 수(number of attention heads)
    "n_layers": 32,             # 레이어 수(number of layers)
    "hidden_dim": 14_336,       # FeedForward의 중간 차원 크기
    "n_kv_groups": 8,           # 그룹화된 쿼리 어텐션을 위한 키-값 그룹
    "rope_base": 500_000.0,     # RoPE의 "theta" 기본값
    "dtype": torch.bfloat16,    # 메모리 사용량을 줄이기 위한 낮은 정밀도 dtype
    "rope_freq": {              # NEW: RoPE 주파수 스케일링
        "factor": 8.0,
        "low_freq_factor": 1.0,
        "high_freq_factor": 4.0,
        "original_context_length": 8192,
    }
}

- 앞서 코드에서 본 바와 같이, RoPE 방법은 사인곡선 함수(사인과 코사인)를 사용하여 위치 정보를 어텐션 메커니즘에 직접 임베드합니다
- Llama 3.1에서는 추가 구성을 통해 역주파수 계산에 추가 조정을 도입합니다
- 이러한 조정은 서로 다른 주파수 성분이 위치 임베딩에 어떻게 기여하는지에 영향을 미칩니다 (자세한 설명은 다음 기회로 미룹니다)
- 실제로 Llama 3.1 모델을 시도해보겠습니다. 먼저 일부 GPU 메모리를 해제하기 위해 이전 모델을 정리합니다

In [ ]:
# 메모리 해제
del model

gc.collect()  # Python 가비지 수집기 실행

if torch.cuda.is_available():
    torch.cuda.empty_cache()

- 다음으로, 토크나이저를 다운로드합니다
- Llama 3.1 계열이 Llama 3 계열과 구별되므로, [meta-llama/Llama-3.1-8B](https://huggingface.co/meta-llama/Llama-3.1-8B) 저장소를 방문하여 라이선스 조건에 동의해야만 Hugging Face 액세스 토큰이 다운로드에 작동합니다
- 팁: 단순성을 위해 아래에서는 기본 모델만 로드하지만, `"meta-llama/Llama-3.1-8B"`를 `"meta-llama/Llama-3.1-8B-Instruct"`로 교체하여 지시 미세조정 버전도 사용할 수 있습니다

In [ ]:
tokenizer_file_path = hf_hub_download(
    repo_id="meta-llama/Llama-3.1-8B",
    filename="original/tokenizer.model",
    local_dir="Llama-3.1-8B"
)

tokenizer = Tokenizer(tokenizer_file_path)

In [ ]:
model = Llama3Model(LLAMA31_CONFIG_8B)

total_params = sum(p.numel() for p in model.parameters())
print(f"Total number of parameters: {total_params:,}")

In [ ]:
combined_weights = {}

for i in range(1, 5):
    weights_file = hf_hub_download(
        repo_id="meta-llama/Llama-3.1-8B",
        filename=f"model-0000{i}-of-00004.safetensors",
        local_dir="Llama-3.1-8B"
    )
    current_weights = load_file(weights_file)
    combined_weights.update(current_weights)

load_weights_into_llama(model, LLAMA31_CONFIG_8B, combined_weights)
model.to(device);
del combined_weights  # 메모리 해제

In [ ]:
torch.manual_seed(123)

token_ids = generate(
    model=model,
    idx=text_to_token_ids("Every effort", tokenizer).to(device),
    max_new_tokens=25,
    context_size=LLAMA31_CONFIG_8B["context_length"],
    top_k=1,
    temperature=0.
)

print("Output text:\n", token_ids_to_text(token_ids, tokenizer))

&nbsp;
# Llama 3.2 1B

- 이 글을 쓰는 시점에서 Meta AI의 최신 모델은 [여기](https://ai.meta.com/blog/llama-3-2-connect-2024-vision-edge-mobile-devices/)에서 발표된 Llama 3.2 모델입니다
- Llama 3.2 텍스트 모델의 코드는 Llama 3.1과 유사하지만, 모델 크기가 줄어든 점이 다릅니다 (1B와 3B 버전이 있습니다)
- 다른 효율성 조정은 가중치 묶기(weight tying)를 다시 추가한 것입니다 (원래 GPT-2 아키텍처에서 사용된 개념); 여기서는 입력(토큰) 임베딩 레이어와 출력 레이어에서 동일한 가중치 매개변수 값을 재사용합니다
- Llama 3.2 1B의 작은 모델 크기는 많은 모바일 기기에서도 실행될 수 있어 상당히 편리합니다
- Llama 3.1 8B와 Llama 3.2 1B 간의 아키텍처 차이점은 아래 그림에 설명되어 있습니다

<img src="https://sebastianraschka.com/images/LLMs-from-scratch-images/bonus/gpt-to-llama/llama31-to-llama32.webp?1" width="700px">

- 위 그림에서 볼 수 있듯이, Llama 3.1 8B와 Llama 3.2 1B 아키텍처의 주요 차이점은 각각의 크기입니다
- 작은 추가 변경사항은 증가된 RoPE 재조정 인수로, 이는 아래 구성 파일에 반영되어 있습니다

In [ ]:
LLAMA31_CONFIG_8B = {
    "vocab_size": 128_256,      # 어휘 크기(vocabulary size)
    "context_length": 131_072,  # NEW: 더 큰 지원 컨텍스트 길이
    "emb_dim": 4096,            # 임베딩 차원(embedding dimension)
    "n_heads": 32,              # 어텐션 헤드 수(number of attention heads)
    "n_layers": 32,             # 레이어 수(number of layers)
    "hidden_dim": 14_336,       # FeedForward의 중간 차원 크기
    "n_kv_groups": 8,           # 그룹화된 쿼리 어텐션을 위한 키-값 그룹
    "rope_base": 500_000.0,     # RoPE의 "theta" 기본값
    "dtype": torch.bfloat16,    # 메모리 사용량을 줄이기 위한 낮은 정밀도 dtype
    "rope_freq": {              # NEW: RoPE 주파수 스케일링
        "factor": 8.0,
        "low_freq_factor": 1.0,
        "high_freq_factor": 4.0,
        "original_context_length": 8192,
    }
}


LLAMA32_CONFIG_1B = {
    "vocab_size": 128_256,      # 어휘 크기(vocabulary size)
    "context_length": 131_072,  # 컨텍스트 길이(context length)
    "emb_dim": 2048,            # NEW: 임베딩 차원의 절반
    "n_heads": 32,              # 어텐션 헤드 수(number of attention heads)
    "n_layers": 16,             # NEW: 레이어 수의 절반
    "hidden_dim": 8192,         # NEW: FeedForward에서 중간 차원의 거의 절반 크기
    "n_kv_groups": 8,           # 그룹화된 쿼리 어텐션을 위한 키-값 그룹
    "rope_base": 500_000.0,     # RoPE의 "theta" 기본값
    "dtype": torch.bfloat16,    # 메모리 사용량을 줄이기 위한 낮은 정밀도 dtype
    "rope_freq": {              # RoPE 주파수 스케일링
        "factor": 32.0,         # NEW: 재조정 인수 조정
        "low_freq_factor": 1.0,
        "high_freq_factor": 4.0,
        "original_context_length": 8192,
    }
}

- 아래에서는 Llama 3.1 8B 섹션의 코드를 재사용하여 Llama 3.2 1B 모델을 로드할 수 있습니다
- 다시, Llama 3.2 계열이 Llama 3.1 계열과 구별되므로, [meta-llama/Llama-3.2-1B](https://huggingface.co/meta-llama/Llama-3.2-1B) 저장소를 방문하여 라이선스 조건에 동의해야만 Hugging Face 액세스 토큰이 다운로드에 작동합니다
- 팁: 단순성을 위해 아래에서는 기본 모델만 로드하지만, `"meta-llama/Llama-3.2-1B"`를 `"meta-llama/Llama-3.2-1B-Instruct"`로 교체하여 지시 미세조정 버전도 사용할 수 있습니다

In [ ]:
# 메모리 해제
del model


gc.collect()  # Python 가비지 수집기 실행

if torch.cuda.is_available():
    torch.cuda.empty_cache()

In [ ]:
tokenizer_file_path = hf_hub_download(
    repo_id="meta-llama/Llama-3.2-1B",
    filename="original/tokenizer.model",
    local_dir="Llama-3.2-1B"
)

tokenizer = Tokenizer(tokenizer_file_path)

In [ ]:
model = Llama3Model(LLAMA32_CONFIG_1B)

total_params = sum(p.numel() for p in model.parameters())
print(f"Total number of parameters: {total_params:,}")

# 가중치 묶기 고려
total_params_normalized = total_params - model.tok_emb.weight.numel()
print(f"\nTotal number of unique parameters: {total_params_normalized:,}")

In [ ]:
weights_file = hf_hub_download(
    repo_id="meta-llama/Llama-3.2-1B",
    filename="model.safetensors",
    local_dir="Llama-3.2-1B"
)
current_weights = load_file(weights_file)

load_weights_into_llama(model, LLAMA32_CONFIG_1B, current_weights)
model.to(device);
del current_weights  # 메모리 해제

In [ ]:
print("Weight tying:", torch.equal(model.tok_emb.weight, model.out_head.weight))

In [ ]:
torch.manual_seed(123)

token_ids = generate(
    model=model,
    idx=text_to_token_ids("Every effort", tokenizer).to(device),
    max_new_tokens=25,
    context_size=LLAMA32_CONFIG_1B["context_length"],
    top_k=1,
    temperature=0.
)

print("Output text:\n", token_ids_to_text(token_ids, tokenizer))

&nbsp;
# 다음 단계는?

- 이 노트북은 GPT에서 Llama 3.2로의 변환을 완료합니다
- Llama 3.2 코드만 포함된 더 간결한 독립 실행형 노트북에 관심이 있으시면 [standalone-llama32.ipynb](standalone-llama32.ipynb) 노트북을 확인하세요